In [ ]:
# ============================================================
# Publication-grade 1D baselines for PMA prediction
#
# Input:
#   Filtered row-wise dataframe where each row is one 10-minute window.
#
# Models:
#   1) Ridge regression (linear baseline)
#   2) XGBoost regression (nonlinear tree baseline)
#
# Feature policy requested by the user:
#   feature_cols = [
#       c for c in df_all.columns
#       if c.startswith("feats__")
#       and c != "feats__pna_days"
#       and c != "feats__weight"
#   ]
#
# Experimental protocol:
#   - Restrict the 10-min dataframe to the exact patient cohort used by the
#     completed 120-min CNN main analysis.
#   - Reuse the exact outer-test patient folds saved by the CNN analysis.
#   - Reuse each CNN outer fold's saved patient ordering and stratification
#     bins (outer_train_inner_bins.csv) to reconstruct the same 3 inner folds.
#   - Optuna tunes model hyperparameters only; there is no architecture search.
#   - All imputation and normalization statistics are fitted on the current
#     training fold only.
#   - Patient-balanced preprocessing and sample weights: each patient has equal
#     total contribution regardless of number of 10-min windows.
#   - Missingness indicators are appended to the imputed/normalized features,
#     providing a value+mask analogue to the 2-channel CNN input.
#   - Model selection uses mean inner-validation patient-level MAE.
#   - Final models are refit on all outer-train patients and evaluated once on
#     outer-test patients.
#   - Ridge is deterministic and refit once. XGBoost uses a 3-seed final
#     ensemble, matching the main CNN robustness protocol.
#   - Complete OOF window-level and patient-level predictions are saved.
#
# Important:
#   This is the full-cohort baseline. It does NOT load or remove the top 10%
#   OOF-error patients.
# ============================================================

from __future__ import annotations

import gc
import hashlib
import json
import math
import os
import pickle
import random
import warnings
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Mapping, Optional, Sequence, Tuple

import joblib
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.model_selection import StratifiedKFold

try:
    import xgboost as xgb
    from xgboost import XGBRegressor
except ImportError as exc:
    raise ImportError(
        "xgboost is required. Install it in the active environment, e.g. "
        "`pip install xgboost`."
    ) from exc

warnings.filterwarnings("default")


# ============================================================
# 0. Configuration
# ============================================================
SEED = 42
PROJECT_ROOT = "/mnt/home/qinqiu/thesis"

# Filtered 10-minute row-wise dataframe created by Filtered_data.ipynb.
DATA_PKL = os.path.join(
    PROJECT_ROOT,
    "data",
    "data_patients_filtered__no_los_nec__ga_le_35__no_intubation.pkl",
)

ID_COL = "ids__uid"
TARGET_COL = "target__pma_w"
TIME_COL = "timestamp"  # used only to preserve temporal row order

# Exactly the requested feature rule.
FEATURE_PREFIX = "feats__"
EXCLUDED_FEATURES = {
    "feats__pna_days",  # deterministic relationship with PMA/GA
    "feats__weight",    # repeated/current weight forbidden by study design
}
STATIC_FEATURES = ["feats__ga_w", "feats__sex", "feats__bw"]

# Completed full-cohort CNN experiment. No top10-removed suffix here.
MAIN_CNN_EXP_ROOT = os.path.join(
    PROJECT_ROOT,
    "result_nested",
    "nested_cv_120min_stride40min_v7_dataset_reuse_patientbalanced_resume100",
)
MAIN_OUTER_ASSIGNMENT_CSV = os.path.join(
    MAIN_CNN_EXP_ROOT,
    "outer_fold_assignment_long.csv",
)
MAIN_CNN_PATIENT_OOF_CSV = os.path.join(
    MAIN_CNN_EXP_ROOT,
    "FINAL_nested_oof_patient_predictions.csv",
)

EXP_ROOT = os.path.join(
    PROJECT_ROOT,
    "result_baseline",
    "baseline_1d_10min_ridge_xgboost_nestedcv_optuna",
)
os.makedirs(EXP_ROOT, exist_ok=True)

OUTER_SPLITS = 5
INNER_SPLITS = 3

N_TRIALS_PER_OUTER: Dict[str, int] = {
    "ridge": 30,
    "xgboost": 100,
}
MODELS_TO_RUN: Tuple[str, ...] = ("ridge", "xgboost")

FINAL_REFIT_SEEDS: Dict[str, List[int]] = {
    "ridge": [42],
    "xgboost": [42, 123, 2026],
}

# XGBoost device configuration.
XGB_DEVICE = "cuda"  # change to "cpu" if GPU XGBoost is unavailable
XGB_N_JOBS = 4

# Missing-data handling.
INCLUDE_MISSING_INDICATORS = True
PATIENT_BALANCED_IMPUTATION = True
PATIENT_BALANCED_NORMALIZATION = True
PATIENT_BALANCED_TRAINING = True

# Exact split reuse. Main analysis should have produced these files.
REQUIRE_MAIN_CNN_OUTER_FOLDS = True
REQUIRE_MAIN_CNN_INNER_BINS = True

# Prepared matrices are cached as .npy and memory-mapped during fitting.
# This saves substantial preprocessing time across Optuna trials, but can use
# several GB of disk because the 10-min dataframe contains many rows.
CACHE_PREPARED_SPLITS = True

SMOKE_TEST = False
if SMOKE_TEST:
    N_TRIALS_PER_OUTER = {"ridge": 2, "xgboost": 2}

CODE_VERSION = "baseline_1d_10min_nestedcv_v1"
PREPROCESSING_MODE = (
    "training_fold_only_patient_balanced_patient_median_imputation_"
    "patient_balanced_featurewise_zscore_missing_indicators"
)


# ============================================================
# 1. Reproducibility and atomic I/O
# ============================================================
def seed_everything(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)


def cleanup() -> None:
    gc.collect()


def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return float(value)
    if isinstance(value, Path):
        return str(value)
    return value


def stable_hash(payload: Any) -> str:
    encoded = json.dumps(
        json_safe(payload),
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=True,
    ).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest()


def atomic_json_dump(payload: Any, path: str) -> None:
    path = os.path.abspath(path)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(json_safe(payload), f, indent=2, sort_keys=True)
    os.replace(tmp, path)


def atomic_npy_save(array: np.ndarray, path: str) -> None:
    path = os.path.abspath(path)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = path + ".tmp.npy"
    np.save(tmp, array)
    os.replace(tmp, path)


def atomic_joblib_dump(obj: Any, path: str) -> None:
    path = os.path.abspath(path)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = path + ".tmp"
    joblib.dump(obj, tmp)
    os.replace(tmp, path)


def read_json(path: str) -> Dict[str, Any]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def normalize_pid_text(value: Any) -> str:
    text = str(value)
    if text.endswith(".0"):
        maybe_int = text[:-2]
        if maybe_int.replace("-", "").isdigit():
            return maybe_int
    return text


seed_everything(SEED)


# ============================================================
# 2. Load filtered 10-min dataframe and select 1D features
# ============================================================
if not os.path.isfile(DATA_PKL):
    raise FileNotFoundError(f"1D filtered dataframe not found: {DATA_PKL}")
if not os.path.isfile(MAIN_OUTER_ASSIGNMENT_CSV):
    raise FileNotFoundError(
        "Main CNN outer-fold assignment not found:\n"
        f"{MAIN_OUTER_ASSIGNMENT_CSV}"
    )

print("DATA_PKL:", DATA_PKL)
print("MAIN_CNN_EXP_ROOT:", MAIN_CNN_EXP_ROOT)
print("EXP_ROOT:", EXP_ROOT)
print("XGBoost version:", xgb.__version__)
print("XGB_DEVICE:", XGB_DEVICE)

with open(DATA_PKL, "rb") as f:
    df_all = pickle.load(f)

if not isinstance(df_all, pd.DataFrame):
    raise TypeError(f"Expected pandas DataFrame, got {type(df_all)}")

required_columns = {ID_COL, TARGET_COL}
missing_required = required_columns - set(df_all.columns)
if missing_required:
    raise KeyError(f"Missing required dataframe columns: {sorted(missing_required)}")

feature_cols = [
    c
    for c in df_all.columns
    if c.startswith(FEATURE_PREFIX) and c not in EXCLUDED_FEATURES
]
if not feature_cols:
    raise ValueError("No 1D feature columns were selected.")
if any(c in feature_cols for c in EXCLUDED_FEATURES):
    raise RuntimeError("An excluded feature was unexpectedly selected.")

missing_static = [c for c in STATIC_FEATURES if c not in feature_cols]
if missing_static:
    raise ValueError(f"Missing required static features: {missing_static}")

# Keep only columns needed by this experiment. Numeric conversion is explicit;
# unexpected nonnumeric strings become NaN and are handled inside each fold.
keep_cols = [ID_COL, TARGET_COL] + feature_cols
if TIME_COL in df_all.columns:
    keep_cols.append(TIME_COL)

df = df_all[keep_cols].copy()
del df_all
cleanup()

df[ID_COL] = df[ID_COL].map(normalize_pid_text)
df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
for col in feature_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

rows_before_required_drop = len(df)
df = df[df[ID_COL].notna() & df[TARGET_COL].notna()].copy()
if not np.isfinite(df[TARGET_COL].to_numpy(dtype=float)).all():
    raise ValueError("Non-finite PMA targets remain after filtering.")
print(
    "Dropped rows with missing patient ID/target:",
    rows_before_required_drop - len(df),
)

if TIME_COL in df.columns:
    df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce")
    # NaT values sort last within a patient; their original order is kept stable.
    df = df.sort_values([ID_COL, TIME_COL], kind="stable").reset_index(drop=True)
else:
    warnings.warn(
        f"{TIME_COL!r} is absent; preserving current row order within patients."
    )
    df = df.sort_values([ID_COL], kind="stable").reset_index(drop=True)

# Every 10-min row gets a stable within-patient index used in OOF output files.
df["__window_idx"] = df.groupby(ID_COL, sort=False).cumcount().astype(np.int32)

pd.DataFrame({"feature": feature_cols}).to_csv(
    os.path.join(EXP_ROOT, "features_for_1d_baseline.csv"), index=False
)
pd.DataFrame(
    {
        "column_index": np.arange(
            len(feature_cols) * (2 if INCLUDE_MISSING_INDICATORS else 1)
        ),
        "source": (
            ["normalized_imputed_value"] * len(feature_cols)
            + (["missing_indicator"] * len(feature_cols) if INCLUDE_MISSING_INDICATORS else [])
        ),
        "feature": feature_cols
        + (feature_cols if INCLUDE_MISSING_INDICATORS else []),
    }
).to_csv(os.path.join(EXP_ROOT, "input_column_map_1d.csv"), index=False)

print("Loaded filtered dataframe:", df.shape)
print("Candidate patients before CNN-cohort alignment:", df[ID_COL].nunique())
print("Selected 1D features:", len(feature_cols))
print("Nominal final input dimension:", len(feature_cols) * (2 if INCLUDE_MISSING_INDICATORS else 1))


# ============================================================
# 3. Align to exact full-cohort CNN patients and outer folds
# ============================================================
def load_main_outer_test_assignments() -> Tuple[pd.DataFrame, Dict[int, List[str]]]:
    assignment = pd.read_csv(MAIN_OUTER_ASSIGNMENT_CSV)
    required = {"pid", "outer_fold", "role"}
    missing = required - set(assignment.columns)
    if missing:
        raise ValueError(
            f"Main fold assignment CSV is missing columns: {sorted(missing)}"
        )

    assignment = assignment.copy()
    assignment["pid"] = assignment["pid"].map(normalize_pid_text)
    test_rows = assignment[assignment["role"] == "outer_test"].copy()
    if len(test_rows) == 0:
        raise ValueError("No outer_test rows found in main fold assignment CSV.")

    test_rows["outer_fold"] = pd.to_numeric(
        test_rows["outer_fold"], errors="raise"
    ).astype(int)

    counts = test_rows.groupby("pid").size()
    if not (counts == 1).all():
        bad = counts[counts != 1]
        raise RuntimeError(
            "Each CNN patient must appear exactly once as outer_test. "
            f"Bad counts: {bad.head().to_dict()}"
        )

    observed_folds = sorted(test_rows["outer_fold"].unique().tolist())
    expected_folds = list(range(1, OUTER_SPLITS + 1))
    if observed_folds != expected_folds:
        raise RuntimeError(
            f"Unexpected outer folds: {observed_folds}; expected {expected_folds}"
        )

    outer_sets: Dict[int, List[str]] = {}
    for fold in expected_folds:
        outer_sets[fold] = test_rows.loc[
            test_rows["outer_fold"] == fold, "pid"
        ].tolist()
        if not outer_sets[fold]:
            raise RuntimeError(f"Outer fold {fold} has no test patients.")

    return test_rows[["pid", "outer_fold"]].copy(), outer_sets


main_test_assignment, outer_test_sets = load_main_outer_test_assignments()
main_cohort_pids = set(main_test_assignment["pid"].tolist())
source_pids = set(df[ID_COL].unique().tolist())

missing_from_1d = sorted(main_cohort_pids - source_pids)
extra_in_1d = sorted(source_pids - main_cohort_pids)
if missing_from_1d:
    raise RuntimeError(
        "The 10-min dataframe is missing patients used by the main CNN cohort: "
        f"{missing_from_1d[:30]}"
    )

# Extra patients can exist because the 120-min superwindow QC may remove some
# people. They are deliberately excluded from this paired baseline.
if extra_in_1d:
    pd.DataFrame(
        {"pid": extra_in_1d, "reason": "not_in_completed_main_CNN_cohort"}
    ).to_csv(
        os.path.join(EXP_ROOT, "patients_excluded_to_match_main_CNN_cohort.csv"),
        index=False,
    )

df = df[df[ID_COL].isin(main_cohort_pids)].copy().reset_index(drop=True)

# Verify static covariates are constant within each patient, ignoring NaN.
static_variation_rows: List[Dict[str, Any]] = []
for col in STATIC_FEATURES:
    n_unique = df.groupby(ID_COL, sort=False)[col].nunique(dropna=True)
    bad = n_unique[n_unique > 1]
    for pid, count in bad.items():
        static_variation_rows.append(
            {"pid": pid, "feature": col, "n_unique_nonmissing": int(count)}
        )
if static_variation_rows:
    pd.DataFrame(static_variation_rows).to_csv(
        os.path.join(EXP_ROOT, "invalid_varying_static_features.csv"), index=False
    )
    raise ValueError(
        "One or more declared static features vary within a patient. "
        "See invalid_varying_static_features.csv."
    )

# Patient summary from native 10-min targets.
patient_df = (
    df.groupby(ID_COL, sort=False)
    .agg(
        n_windows=(TARGET_COL, "size"),
        target_mean=(TARGET_COL, "mean"),
        target_median=(TARGET_COL, "median"),
    )
    .reset_index()
    .rename(columns={ID_COL: "pid"})
)
patient_df["pid"] = patient_df["pid"].astype(str)
patient_df = patient_df.sort_values("pid", kind="stable").reset_index(drop=True)
patient_df.to_csv(os.path.join(EXP_ROOT, "patient_summary_before_cv.csv"), index=False)

if set(patient_df["pid"]) != main_cohort_pids:
    raise RuntimeError("Final 1D patient cohort does not exactly match main CNN cohort.")
if len(patient_df) < OUTER_SPLITS:
    raise ValueError("Not enough patients for outer CV.")

# Save reused test assignments in a compact form.
main_test_assignment.sort_values(["outer_fold", "pid"]).to_csv(
    os.path.join(EXP_ROOT, "reused_main_outer_test_assignments.csv"), index=False
)

print("Final paired patient cohort:", len(patient_df))
print("Final 10-min windows:", len(df))
for fold, pids in outer_test_sets.items():
    print(f"Reused outer fold {fold}: {len(pids)} test patients")


# ============================================================
# 4. Dense arrays and fast patient row lookup
# ============================================================
X_RAW = df[feature_cols].to_numpy(dtype=np.float32, copy=True)
Y_ALL = df[TARGET_COL].to_numpy(dtype=np.float32, copy=True)
PID_ALL = df[ID_COL].astype(str).to_numpy()
WINDOW_IDX_ALL = df["__window_idx"].to_numpy(dtype=np.int32, copy=True)

patient_row_indices: Dict[str, np.ndarray] = {
    str(pid): group.index.to_numpy(dtype=np.int64)
    for pid, group in df.groupby(ID_COL, sort=False)
}

# Per-patient feature medians are independent of cross-patient folds and can be
# precomputed safely. A fold's imputation vector is the median across only its
# training patients' per-patient medians, giving every patient equal influence.
patient_feature_medians: Dict[str, np.ndarray] = {}
for pid, indices in patient_row_indices.items():
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        med = np.nanmedian(X_RAW[indices].astype(np.float64), axis=0)
    patient_feature_medians[pid] = med.astype(np.float32)

# The dataframe is no longer required after arrays/indices are built.
del df
cleanup()


def rows_for_patients(pids: Sequence[str]) -> np.ndarray:
    parts = [patient_row_indices[str(pid)] for pid in pids]
    if not parts:
        return np.empty((0,), dtype=np.int64)
    return np.concatenate(parts).astype(np.int64, copy=False)


def patient_balanced_sample_weights(window_pids: np.ndarray) -> np.ndarray:
    pids = np.asarray(window_pids).astype(str)
    if len(pids) == 0:
        return np.empty((0,), dtype=np.float64)
    counts = Counter(pids.tolist())
    weights = np.asarray([1.0 / counts[pid] for pid in pids], dtype=np.float64)
    totals = {pid: float(weights[pids == pid].sum()) for pid in counts}
    if not np.allclose(list(totals.values()), 1.0, rtol=0.0, atol=1e-10):
        raise RuntimeError("Patient-balanced sample weights failed validation.")
    return weights


# ============================================================
# 5. Reconstruct exact CNN inner folds
# ============================================================
def make_strat_bins(
    y_values: Sequence[float],
    n_splits: int,
    max_bins: int = 8,
) -> Tuple[np.ndarray, int]:
    y_values = np.asarray(y_values, dtype=float)
    if not np.isfinite(y_values).all():
        raise ValueError("Non-finite patient targets passed to stratification.")

    for n_bins in range(max_bins, 1, -1):
        try:
            bins = pd.qcut(
                y_values,
                q=n_bins,
                labels=False,
                duplicates="drop",
            )
            bins = np.asarray(bins, dtype=int)
            counts = pd.Series(bins).value_counts()
            if counts.min() >= n_splits:
                return bins, n_bins
        except Exception:
            continue
    return np.zeros(len(y_values), dtype=int), 1


def reconstruct_inner_splits(
    outer_fold: int,
    outer_train_pids: Sequence[str],
) -> Tuple[List[Tuple[int, List[str], List[str]]], pd.DataFrame]:
    inner_bins_csv = os.path.join(
        MAIN_CNN_EXP_ROOT,
        f"outer_fold_{outer_fold:02d}",
        "outer_train_inner_bins.csv",
    )

    expected_set = {str(x) for x in outer_train_pids}

    if os.path.isfile(inner_bins_csv):
        work = pd.read_csv(inner_bins_csv)
        required = {"pid", "inner_strat_bin"}
        missing = required - set(work.columns)
        if missing:
            raise ValueError(
                f"{inner_bins_csv} is missing columns: {sorted(missing)}"
            )
        work = work.copy()
        work["pid"] = work["pid"].map(normalize_pid_text)
        work["inner_strat_bin"] = pd.to_numeric(
            work["inner_strat_bin"], errors="raise"
        ).astype(int)

        observed_set = set(work["pid"])
        if observed_set != expected_set:
            raise RuntimeError(
                f"Outer {outer_fold}: saved CNN inner-bin cohort mismatch. "
                f"Missing={sorted(expected_set - observed_set)[:20]}, "
                f"unexpected={sorted(observed_set - expected_set)[:20]}"
            )
        if work["pid"].duplicated().any():
            raise RuntimeError(
                f"Outer {outer_fold}: duplicate patients in {inner_bins_csv}"
            )
        source = "reused_main_CNN_inner_bins_and_order"
    else:
        if REQUIRE_MAIN_CNN_INNER_BINS:
            raise FileNotFoundError(
                "Saved main-CNN inner-bin file not found:\n"
                f"{inner_bins_csv}"
            )
        # Explicit fallback: native 1D target means. This does not guarantee
        # identical inner splits and is disabled by default.
        work = patient_df[patient_df["pid"].isin(expected_set)].copy()
        work = work.sort_values("pid", kind="stable").reset_index(drop=True)
        bins, used_bins = make_strat_bins(
            work["target_mean"].values,
            n_splits=INNER_SPLITS,
            max_bins=8,
        )
        work["inner_strat_bin"] = bins
        work["target_mean_main_CNN"] = np.nan
        source = f"fallback_1D_native_target_bins_{used_bins}"

    inner_cv = StratifiedKFold(
        n_splits=INNER_SPLITS,
        shuffle=True,
        random_state=SEED + outer_fold,
    )

    splits: List[Tuple[int, List[str], List[str]]] = []
    for inner_fold, (tr_idx, val_idx) in enumerate(
        inner_cv.split(work["pid"].values, work["inner_strat_bin"].values),
        start=1,
    ):
        train_pids = work.iloc[tr_idx]["pid"].astype(str).tolist()
        val_pids = work.iloc[val_idx]["pid"].astype(str).tolist()
        splits.append((inner_fold, train_pids, val_pids))

    work = work.copy()
    work["split_source"] = source
    print(
        f"Outer {outer_fold}: reconstructed inner folds from {source}; "
        f"sizes={[(len(t), len(v)) for _, t, v in splits]}"
    )
    return splits, work


# ============================================================
# 6. Training-fold-only preprocessing and persistent cache
# ============================================================
@dataclass(frozen=True)
class PreprocessStats:
    imputation_values: np.ndarray  # (F,)
    feature_mean: np.ndarray       # (F,)
    feature_std: np.ndarray        # (F,)
    all_missing_in_training: np.ndarray  # bool (F,)


def compute_imputation_values(train_pids: Sequence[str]) -> Tuple[np.ndarray, np.ndarray]:
    if not train_pids:
        raise ValueError("Empty train_pids passed to imputation.")

    patient_medians = np.stack(
        [patient_feature_medians[str(pid)] for pid in train_pids], axis=0
    ).astype(np.float64)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        if PATIENT_BALANCED_IMPUTATION:
            values = np.nanmedian(patient_medians, axis=0)
        else:
            train_rows = rows_for_patients(train_pids)
            values = np.nanmedian(X_RAW[train_rows].astype(np.float64), axis=0)

    all_missing = ~np.isfinite(values)
    # No test/validation information is used. If a feature is absent from every
    # training patient, the imputed raw value is set to zero; after z-scoring it
    # becomes zero and its missing indicator remains available.
    values[all_missing] = 0.0
    return values.astype(np.float32), all_missing.astype(bool)


def compute_preprocess_stats(
    train_pids: Sequence[str],
    train_rows: np.ndarray,
    train_window_pids: np.ndarray,
) -> PreprocessStats:
    impute, all_missing = compute_imputation_values(train_pids)

    raw = X_RAW[train_rows].astype(np.float32, copy=True)
    missing = ~np.isfinite(raw)
    raw[missing] = np.broadcast_to(impute, raw.shape)[missing]

    if not np.isfinite(raw).all():
        raise RuntimeError("Non-finite values remain after training-fold imputation.")

    if PATIENT_BALANCED_NORMALIZATION:
        weights = patient_balanced_sample_weights(train_window_pids)
    else:
        weights = np.ones(len(raw), dtype=np.float64)

    weight_sum = float(weights.sum())
    if weight_sum <= 0:
        raise ValueError("Normalization weights sum to zero.")

    raw64 = raw.astype(np.float64, copy=False)
    mean = np.sum(raw64 * weights[:, None], axis=0) / weight_sum
    var = np.sum(
        ((raw64 - mean.reshape(1, -1)) ** 2) * weights[:, None], axis=0
    ) / weight_sum
    std = np.sqrt(np.maximum(var, 1e-12))
    std = np.maximum(std, 1e-6)

    # Features completely missing in training carry no value information.
    # Force a neutral normalization; missing indicators still encode absence.
    mean[all_missing] = 0.0
    std[all_missing] = 1.0

    return PreprocessStats(
        imputation_values=impute.astype(np.float32),
        feature_mean=mean.astype(np.float32),
        feature_std=std.astype(np.float32),
        all_missing_in_training=all_missing.astype(bool),
    )


def transform_rows(rows: np.ndarray, stats: PreprocessStats) -> np.ndarray:
    raw = X_RAW[rows].astype(np.float32, copy=True)
    missing = ~np.isfinite(raw)
    raw[missing] = np.broadcast_to(stats.imputation_values, raw.shape)[missing]

    values = (
        raw - stats.feature_mean.reshape(1, -1)
    ) / stats.feature_std.reshape(1, -1)
    values = values.astype(np.float32)

    if not np.isfinite(values).all():
        raise RuntimeError("Non-finite values remain after 1D normalization.")

    parts = [values]
    if INCLUDE_MISSING_INDICATORS:
        parts.append(missing.astype(np.float32))
    return np.concatenate(parts, axis=1).astype(np.float32, copy=False)


@dataclass(frozen=True)
class PreparedSplit:
    cache_dir: str
    split_hash: str

    def path(self, name: str) -> str:
        return os.path.join(self.cache_dir, f"{name}.npy")

    def load(self, name: str, mmap: bool = True) -> np.ndarray:
        return np.load(self.path(name), mmap_mode="r" if mmap else None)

    @property
    def meta(self) -> Dict[str, Any]:
        return read_json(os.path.join(self.cache_dir, "meta.json"))


def ensure_prepared_split(
    cache_dir: str,
    split_name: str,
    train_pids: Sequence[str],
    eval_pids: Sequence[str],
    parent_identity: Mapping[str, Any],
) -> PreparedSplit:
    os.makedirs(cache_dir, exist_ok=True)

    split_identity = {
        **dict(parent_identity),
        "split_name": split_name,
        "train_pids": [str(x) for x in train_pids],
        "eval_pids": [str(x) for x in eval_pids],
        "preprocessing": PREPROCESSING_MODE,
        "features": feature_cols,
        "excluded_features": sorted(EXCLUDED_FEATURES),
        "include_missing_indicators": INCLUDE_MISSING_INDICATORS,
    }
    split_hash = stable_hash(split_identity)
    meta_path = os.path.join(cache_dir, "meta.json")

    required_arrays = [
        "X_train",
        "y_train",
        "w_train",
        "train_pids",
        "train_window_idx",
        "X_eval",
        "y_eval",
        "eval_pids",
        "eval_window_idx",
        "imputation_values",
        "feature_mean",
        "feature_std",
        "all_missing_in_training",
    ]
    required_paths = [os.path.join(cache_dir, f"{name}.npy") for name in required_arrays]

    if CACHE_PREPARED_SPLITS and os.path.isfile(meta_path):
        meta = read_json(meta_path)
        if meta.get("split_hash") != split_hash:
            raise RuntimeError(
                f"Prepared split cache mismatch at {cache_dir}. "
                "Use a new EXP_ROOT or delete this incompatible cache."
            )
        missing_paths = [p for p in required_paths if not os.path.isfile(p)]
        if not missing_paths:
            print(f"Using prepared cache: {split_name}")
            return PreparedSplit(cache_dir=cache_dir, split_hash=split_hash)

    print(f"Preparing split cache: {split_name}")
    train_rows = rows_for_patients(train_pids)
    eval_rows = rows_for_patients(eval_pids)

    if len(train_rows) == 0 or len(eval_rows) == 0:
        raise ValueError(f"{split_name}: empty train or evaluation rows.")

    train_window_pids = PID_ALL[train_rows].astype(str)
    eval_window_pids = PID_ALL[eval_rows].astype(str)

    stats = compute_preprocess_stats(
        train_pids=train_pids,
        train_rows=train_rows,
        train_window_pids=train_window_pids,
    )

    X_train = transform_rows(train_rows, stats)
    X_eval = transform_rows(eval_rows, stats)
    y_train = Y_ALL[train_rows].astype(np.float32, copy=True)
    y_eval = Y_ALL[eval_rows].astype(np.float32, copy=True)
    w_train = patient_balanced_sample_weights(train_window_pids)
    train_window_idx = WINDOW_IDX_ALL[train_rows].astype(np.int32, copy=True)
    eval_window_idx = WINDOW_IDX_ALL[eval_rows].astype(np.int32, copy=True)

    if X_train.shape[1] != X_eval.shape[1]:
        raise RuntimeError(f"{split_name}: train/eval input dimensions differ.")
    if not np.isfinite(y_train).all() or not np.isfinite(y_eval).all():
        raise RuntimeError(f"{split_name}: non-finite targets.")

    arrays = {
        "X_train": X_train,
        "y_train": y_train,
        "w_train": w_train,
        "train_pids": np.asarray(train_window_pids, dtype=str),
        "train_window_idx": train_window_idx,
        "X_eval": X_eval,
        "y_eval": y_eval,
        "eval_pids": np.asarray(eval_window_pids, dtype=str),
        "eval_window_idx": eval_window_idx,
        "imputation_values": stats.imputation_values,
        "feature_mean": stats.feature_mean,
        "feature_std": stats.feature_std,
        "all_missing_in_training": stats.all_missing_in_training.astype(np.uint8),
    }
    for name, arr in arrays.items():
        atomic_npy_save(np.asarray(arr), os.path.join(cache_dir, f"{name}.npy"))

    meta = {
        "split_hash": split_hash,
        "split_identity": split_identity,
        "n_train_patients": len(train_pids),
        "n_eval_patients": len(eval_pids),
        "n_train_windows": int(len(y_train)),
        "n_eval_windows": int(len(y_eval)),
        "input_dim": int(X_train.shape[1]),
        "n_features_all_missing_in_training": int(
            stats.all_missing_in_training.sum()
        ),
        "all_missing_feature_names": [
            feature_cols[i]
            for i in np.flatnonzero(stats.all_missing_in_training)
        ],
        "train_matrix_size_gib": float(X_train.nbytes / (1024 ** 3)),
        "eval_matrix_size_gib": float(X_eval.nbytes / (1024 ** 3)),
    }
    atomic_json_dump(meta, meta_path)

    del X_train, X_eval, y_train, y_eval, w_train
    cleanup()
    return PreparedSplit(cache_dir=cache_dir, split_hash=split_hash)


# ============================================================
# 7. Evaluation utilities
# ============================================================
def safe_r2(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    if len(y_true) < 2 or np.allclose(y_true, y_true[0]):
        return float("nan")
    return float(r2_score(y_true, y_pred))


def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    if len(y_true) == 0:
        return {"mae": np.nan, "rmse": np.nan, "r2": np.nan, "n": 0}
    err = y_pred - y_true
    return {
        "mae": float(np.mean(np.abs(err))),
        "rmse": float(np.sqrt(np.mean(err ** 2))),
        "r2": safe_r2(y_true, y_pred),
        "n": int(len(y_true)),
    }


def patient_prediction_table(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    pids: np.ndarray,
    outer_fold: int,
    model_name: str,
    seed: int,
) -> pd.DataFrame:
    frame = pd.DataFrame(
        {
            "pid": np.asarray(pids).astype(str),
            "true_window": np.asarray(y_true, dtype=np.float32),
            "pred_window": np.asarray(y_pred, dtype=np.float32),
        }
    )
    patient = (
        frame.groupby("pid", as_index=False)
        .agg(
            true_patient=("true_window", "mean"),
            pred_patient=("pred_window", "mean"),
            true_window_std=("true_window", "std"),
            pred_window_std=("pred_window", "std"),
            n_windows=("true_window", "size"),
        )
    )
    patient[["true_window_std", "pred_window_std"]] = patient[
        ["true_window_std", "pred_window_std"]
    ].fillna(0.0)
    patient["error_patient"] = patient["pred_patient"] - patient["true_patient"]
    patient["abs_error"] = np.abs(patient["error_patient"])
    patient["sq_error"] = patient["error_patient"] ** 2
    patient["outer_fold"] = int(outer_fold)
    patient["model"] = model_name
    patient["seed"] = int(seed)
    return patient


def metrics_from_prepared_eval(
    prepared: PreparedSplit,
    predictions: np.ndarray,
) -> Tuple[Dict[str, float], Dict[str, float], pd.DataFrame]:
    y_eval = np.asarray(prepared.load("y_eval"), dtype=np.float32)
    eval_pids = np.asarray(prepared.load("eval_pids"), dtype=str)
    pred = np.asarray(predictions, dtype=np.float32)
    if len(pred) != len(y_eval):
        raise ValueError("Prediction length mismatch.")

    window_metrics = regression_metrics(y_eval, pred)
    patient = patient_prediction_table(
        y_eval,
        pred,
        eval_pids,
        outer_fold=-1,
        model_name="inner_validation",
        seed=-1,
    )
    patient_metrics = regression_metrics(
        patient["true_patient"].values,
        patient["pred_patient"].values,
    )
    return window_metrics, patient_metrics, patient


# ============================================================
# 8. Fixed model families and Optuna hyperparameter spaces
# ============================================================
def sample_model_params(model_name: str, trial: optuna.Trial) -> Dict[str, Any]:
    if model_name == "ridge":
        return {
            "alpha": trial.suggest_float("alpha", 1e-5, 1e5, log=True),
            "tol": trial.suggest_categorical("tol", [1e-3, 1e-4, 1e-5]),
        }

    if model_name == "xgboost":
        return {
            "n_estimators": trial.suggest_int("n_estimators", 200, 1800, step=100),
            "max_depth": trial.suggest_int("max_depth", 2, 10),
            "learning_rate": trial.suggest_float(
                "learning_rate", 0.005, 0.20, log=True
            ),
            "min_child_weight": trial.suggest_float(
                "min_child_weight", 0.25, 40.0, log=True
            ),
            "subsample": trial.suggest_float("subsample", 0.60, 1.00),
            "colsample_bytree": trial.suggest_float(
                "colsample_bytree", 0.40, 1.00
            ),
            "gamma": trial.suggest_float("gamma", 0.0, 3.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 20.0, log=True),
            "reg_lambda": trial.suggest_float(
                "reg_lambda", 1e-3, 100.0, log=True
            ),
            "max_bin": trial.suggest_categorical("max_bin", [128, 256, 512]),
        }

    raise ValueError(f"Unknown model: {model_name}")


def xgb_version_major() -> int:
    try:
        return int(str(xgb.__version__).split(".")[0])
    except Exception:
        return 2


def build_model(
    model_name: str,
    params: Mapping[str, Any],
    seed: int,
):
    if model_name == "ridge":
        return Ridge(
            alpha=float(params["alpha"]),
            fit_intercept=True,
            solver="lsqr",
            tol=float(params.get("tol", 1e-4)),
            max_iter=20000,
        )

    if model_name == "xgboost":
        kwargs: Dict[str, Any] = {
            **dict(params),
            "objective": "reg:squarederror",
            "eval_metric": "mae",
            "random_state": int(seed),
            "n_jobs": int(XGB_N_JOBS),
            "tree_method": "hist",
            "verbosity": 0,
        }
        if XGB_DEVICE == "cuda":
            if xgb_version_major() >= 2:
                kwargs["device"] = "cuda"
            else:
                kwargs["tree_method"] = "gpu_hist"
                kwargs["predictor"] = "gpu_predictor"
        elif XGB_DEVICE != "cpu":
            raise ValueError("XGB_DEVICE must be 'cuda' or 'cpu'.")
        return XGBRegressor(**kwargs)

    raise ValueError(f"Unknown model: {model_name}")


def fit_model(
    model_name: str,
    params: Mapping[str, Any],
    prepared: PreparedSplit,
    seed: int,
):
    X_train = prepared.load("X_train")
    y_train = np.asarray(prepared.load("y_train"), dtype=np.float32)
    w_train = np.asarray(prepared.load("w_train"), dtype=np.float64)

    model = build_model(model_name, params, seed)
    model.fit(X_train, y_train, sample_weight=w_train)
    return model


# ============================================================
# 9. Experiment identity
# ============================================================
if "__file__" in globals() and os.path.isfile(__file__):
    with open(__file__, "rb") as f:
        SCRIPT_SHA256 = hashlib.sha256(f.read()).hexdigest()
else:
    SCRIPT_SHA256 = "unavailable"

stat = os.stat(DATA_PKL)
dataset_fingerprint = {
    "absolute_path": os.path.abspath(DATA_PKL),
    "size_bytes": int(stat.st_size),
    "mtime_ns": int(stat.st_mtime_ns),
}
assignment_stat = os.stat(MAIN_OUTER_ASSIGNMENT_CSV)
assignment_fingerprint = {
    "absolute_path": os.path.abspath(MAIN_OUTER_ASSIGNMENT_CSV),
    "size_bytes": int(assignment_stat.st_size),
    "mtime_ns": int(assignment_stat.st_mtime_ns),
}

run_config = {
    "code_version": CODE_VERSION,
    "script_sha256": SCRIPT_SHA256,
    "dataset_fingerprint": dataset_fingerprint,
    "main_assignment_fingerprint": assignment_fingerprint,
    "main_cnn_exp_root": os.path.abspath(MAIN_CNN_EXP_ROOT),
    "outer_splits": OUTER_SPLITS,
    "inner_splits": INNER_SPLITS,
    "models": list(MODELS_TO_RUN),
    "n_trials_per_outer": N_TRIALS_PER_OUTER,
    "final_refit_seeds": FINAL_REFIT_SEEDS,
    "preprocessing": PREPROCESSING_MODE,
    "patient_balanced_imputation": PATIENT_BALANCED_IMPUTATION,
    "patient_balanced_normalization": PATIENT_BALANCED_NORMALIZATION,
    "patient_balanced_training": PATIENT_BALANCED_TRAINING,
    "include_missing_indicators": INCLUDE_MISSING_INDICATORS,
    "feature_selection": (
        'startswith("feats__") excluding feats__pna_days and feats__weight'
    ),
    "features": feature_cols,
    "excluded_features": sorted(EXCLUDED_FEATURES),
    "static_features": STATIC_FEATURES,
    "n_patients": int(len(patient_df)),
    "n_10min_windows": int(len(Y_ALL)),
    "xgboost_version": xgb.__version__,
    "xgb_device": XGB_DEVICE,
}
EXPERIMENT_HASH = stable_hash(run_config)
run_config["experiment_hash"] = EXPERIMENT_HASH

signature_path = os.path.join(EXP_ROOT, "experiment_signature.json")
if os.path.isfile(signature_path):
    existing = read_json(signature_path)
    if existing.get("experiment_hash") != EXPERIMENT_HASH:
        raise RuntimeError(
            "EXP_ROOT contains an incompatible experiment. Use a new EXP_ROOT "
            "or restore the matching script/configuration."
        )
else:
    atomic_json_dump(run_config, signature_path)
atomic_json_dump(run_config, os.path.join(EXP_ROOT, "run_config.json"))
print("EXPERIMENT_HASH:", EXPERIMENT_HASH)


# ============================================================
# 10. Optuna and prediction aggregation helpers
# ============================================================
def trial_state_counts(study: optuna.Study) -> Dict[str, int]:
    counts = Counter(trial.state.name for trial in study.trials)
    return {
        state: int(counts.get(state, 0))
        for state in ["COMPLETE", "PRUNED", "FAIL", "RUNNING", "WAITING"]
    }


def top_up_study(
    study: optuna.Study,
    objective,
    target_count: int,
) -> None:
    while True:
        counts = trial_state_counts(study)
        counted = counts["COMPLETE"] + counts["PRUNED"]
        remaining = int(target_count - counted)
        print(
            f"Study {study.study_name}: COMPLETE={counts['COMPLETE']}, "
            f"PRUNED={counts['PRUNED']}, FAIL={counts['FAIL']}, "
            f"COUNTED={counted}/{target_count}, REMAINING={remaining}"
        )
        if remaining <= 0:
            break

        before = counted
        catch_types: Tuple[type, ...] = (
            RuntimeError,
            MemoryError,
            xgb.core.XGBoostError,
        )
        study.optimize(
            objective,
            n_trials=remaining,
            gc_after_trial=True,
            catch=catch_types,
        )
        after_counts = trial_state_counts(study)
        after = after_counts["COMPLETE"] + after_counts["PRUNED"]
        if after <= before:
            raise RuntimeError(
                f"Study {study.study_name} made no progress. Inspect failed trials."
            )


def save_study_outputs(study: optuna.Study, output_dir: str) -> None:
    study.trials_dataframe().to_csv(
        os.path.join(output_dir, "optuna_trials.csv"), index=False
    )
    atomic_json_dump(
        {
            "best_trial_number": int(study.best_trial.number),
            "best_value_inner_patient_mae": float(study.best_value),
            "best_params": study.best_params,
            "best_user_attrs": study.best_trial.user_attrs,
            "state_counts": trial_state_counts(study),
        },
        os.path.join(output_dir, "optuna_best.json"),
    )


def make_window_prediction_df(
    prepared: PreparedSplit,
    predictions: np.ndarray,
    outer_fold: int,
    model_name: str,
    seed: int,
) -> pd.DataFrame:
    y_eval = np.asarray(prepared.load("y_eval"), dtype=np.float32)
    eval_pids = np.asarray(prepared.load("eval_pids"), dtype=str)
    eval_window_idx = np.asarray(
        prepared.load("eval_window_idx"), dtype=np.int32
    )
    pred = np.asarray(predictions, dtype=np.float32)
    if len(pred) != len(y_eval):
        raise ValueError("Prediction length mismatch.")

    return pd.DataFrame(
        {
            "pid": eval_pids,
            "window_idx": eval_window_idx,
            "window_id": [
                f"{pid}_{idx}" for pid, idx in zip(eval_pids, eval_window_idx)
            ],
            "true_window": y_eval,
            "pred_window": pred,
            "error_window": pred - y_eval,
            "outer_fold": int(outer_fold),
            "model": model_name,
            "seed": int(seed),
        }
    )


def ensemble_seed_window_predictions(seed_predictions: pd.DataFrame) -> pd.DataFrame:
    if len(seed_predictions) == 0:
        return pd.DataFrame()
    keys = ["pid", "window_idx", "window_id", "outer_fold", "model"]
    true_counts = seed_predictions.groupby(keys)["true_window"].nunique()
    if not (true_counts == 1).all():
        raise RuntimeError("Inconsistent true_window values across final seeds.")
    out = (
        seed_predictions.groupby(keys, as_index=False)
        .agg(
            true_window=("true_window", "first"),
            pred_window_mean=("pred_window", "mean"),
            pred_window_std_across_seeds=("pred_window", "std"),
            n_seeds=("pred_window", "size"),
        )
    )
    out["pred_window_std_across_seeds"] = out[
        "pred_window_std_across_seeds"
    ].fillna(0.0)
    out["error_window"] = out["pred_window_mean"] - out["true_window"]
    out["abs_error_window"] = np.abs(out["error_window"])
    return out


def patient_from_ensemble_windows(window_df: pd.DataFrame) -> pd.DataFrame:
    pat = (
        window_df.groupby(["pid", "outer_fold", "model"], as_index=False)
        .agg(
            true_patient=("true_window", "mean"),
            pred_patient=("pred_window_mean", "mean"),
            pred_patient_std_across_windows=("pred_window_mean", "std"),
            n_windows=("window_idx", "size"),
            n_seeds=("n_seeds", "first"),
        )
    )
    pat["pred_patient_std_across_windows"] = pat[
        "pred_patient_std_across_windows"
    ].fillna(0.0)
    pat["error_patient"] = pat["pred_patient"] - pat["true_patient"]
    pat["abs_error"] = np.abs(pat["error_patient"])
    pat["sq_error"] = pat["error_patient"] ** 2
    return pat


# ============================================================
# 11. Run nested CV
# ============================================================
all_model_outer_windows: Dict[str, List[pd.DataFrame]] = {
    model: [] for model in MODELS_TO_RUN
}
all_model_outer_patients: Dict[str, List[pd.DataFrame]] = {
    model: [] for model in MODELS_TO_RUN
}
outer_summary_rows: List[Dict[str, Any]] = []

all_ordered_pids = patient_df["pid"].astype(str).tolist()

for outer_fold in range(1, OUTER_SPLITS + 1):
    outer_dir = os.path.join(EXP_ROOT, f"outer_fold_{outer_fold:02d}")
    os.makedirs(outer_dir, exist_ok=True)

    outer_test_pids = [str(x) for x in outer_test_sets[outer_fold]]
    outer_test_set = set(outer_test_pids)
    outer_train_pids = [
        pid for pid in all_ordered_pids if pid not in outer_test_set
    ]

    if set(outer_train_pids) & outer_test_set:
        raise RuntimeError(f"Outer fold {outer_fold}: train/test patient overlap.")
    if set(outer_train_pids) | outer_test_set != set(all_ordered_pids):
        raise RuntimeError(f"Outer fold {outer_fold}: incomplete patient partition.")

    outer_identity = {
        "experiment_hash": EXPERIMENT_HASH,
        "outer_fold": outer_fold,
        "outer_train_pids": outer_train_pids,
        "outer_test_pids": outer_test_pids,
    }
    outer_hash = stable_hash(outer_identity)

    print("\n" + "=" * 110)
    print(
        f"OUTER FOLD {outer_fold}/{OUTER_SPLITS}: "
        f"train patients={len(outer_train_pids)}, "
        f"test patients={len(outer_test_pids)}"
    )
    print("=" * 110)

    inner_splits, reused_inner_bins = reconstruct_inner_splits(
        outer_fold, outer_train_pids
    )
    reused_inner_bins.to_csv(
        os.path.join(outer_dir, "reused_or_reconstructed_inner_bins.csv"),
        index=False,
    )

    inner_assignment_rows: List[Dict[str, Any]] = []
    for inner_fold, train_pids, val_pids in inner_splits:
        for pid in train_pids:
            inner_assignment_rows.append(
                {"inner_fold": inner_fold, "pid": pid, "role": "inner_train"}
            )
        for pid in val_pids:
            inner_assignment_rows.append(
                {"inner_fold": inner_fold, "pid": pid, "role": "inner_val"}
            )
    pd.DataFrame(inner_assignment_rows).to_csv(
        os.path.join(outer_dir, "inner_fold_assignment_long.csv"), index=False
    )

    prepared_root = os.path.join(outer_dir, "prepared_1d_10min")
    prepared_inner: List[Tuple[int, PreparedSplit]] = []
    for inner_fold, inner_train_pids, inner_val_pids in inner_splits:
        prepared = ensure_prepared_split(
            cache_dir=os.path.join(
                prepared_root, f"inner_fold_{inner_fold:02d}"
            ),
            split_name=f"outer{outer_fold}_inner{inner_fold}",
            train_pids=inner_train_pids,
            eval_pids=inner_val_pids,
            parent_identity={"outer_hash": outer_hash, "inner_fold": inner_fold},
        )
        prepared_inner.append((inner_fold, prepared))

    prepared_final = ensure_prepared_split(
        cache_dir=os.path.join(prepared_root, "full_outer_train_to_test"),
        split_name=f"outer{outer_fold}_full_train_to_test",
        train_pids=outer_train_pids,
        eval_pids=outer_test_pids,
        parent_identity={"outer_hash": outer_hash, "stage": "final_refit"},
    )

    for model_name in MODELS_TO_RUN:
        model_dir = os.path.join(outer_dir, model_name)
        os.makedirs(model_dir, exist_ok=True)
        complete_marker = os.path.join(model_dir, "MODEL_OUTER_COMPLETE.json")
        ensemble_window_csv = os.path.join(
            model_dir, "outer_test_window_predictions_ensemble.csv"
        )
        ensemble_patient_csv = os.path.join(
            model_dir, "outer_test_patient_predictions_ensemble.csv"
        )

        if os.path.isfile(complete_marker):
            marker = read_json(complete_marker)
            if (
                marker.get("experiment_hash") != EXPERIMENT_HASH
                or marker.get("outer_hash") != outer_hash
                or marker.get("model") != model_name
            ):
                raise RuntimeError(
                    f"Incompatible completion marker: {complete_marker}"
                )
            if not os.path.isfile(ensemble_window_csv) or not os.path.isfile(
                ensemble_patient_csv
            ):
                raise RuntimeError(
                    "Completion marker exists but prediction CSVs are missing: "
                    f"{model_dir}"
                )
            loaded_w = pd.read_csv(ensemble_window_csv, dtype={"pid": str})
            loaded_p = pd.read_csv(ensemble_patient_csv, dtype={"pid": str})
            all_model_outer_windows[model_name].append(loaded_w)
            all_model_outer_patients[model_name].append(loaded_p)
            outer_summary_rows.append(marker["outer_summary"])
            print(
                f"Outer {outer_fold} {model_name}: already complete; loaded outputs."
            )
            continue

        def objective(trial: optuna.Trial) -> float:
            params = sample_model_params(model_name, trial)
            patient_maes: List[float] = []
            patient_rmses: List[float] = []
            patient_r2s: List[float] = []
            window_maes: List[float] = []

            for inner_fold, prepared in prepared_inner:
                run_seed = SEED + outer_fold * 100 + inner_fold
                model = fit_model(model_name, params, prepared, run_seed)
                X_eval = prepared.load("X_eval")
                pred = np.asarray(model.predict(X_eval), dtype=np.float32)
                window_metrics, patient_metrics, _ = metrics_from_prepared_eval(
                    prepared, pred
                )

                patient_maes.append(patient_metrics["mae"])
                patient_rmses.append(patient_metrics["rmse"])
                patient_r2s.append(patient_metrics["r2"])
                window_maes.append(window_metrics["mae"])

                trial.report(float(np.mean(patient_maes)), step=inner_fold)
                del model, pred
                cleanup()

                if trial.should_prune():
                    raise optuna.TrialPruned()

            trial.set_user_attr(
                "inner_val_patient_MAE_mean", float(np.mean(patient_maes))
            )
            trial.set_user_attr(
                "inner_val_patient_MAE_std", float(np.std(patient_maes, ddof=0))
            )
            trial.set_user_attr(
                "inner_val_patient_RMSE_mean", float(np.mean(patient_rmses))
            )
            trial.set_user_attr(
                "inner_val_patient_R2_mean", float(np.nanmean(patient_r2s))
            )
            trial.set_user_attr(
                "inner_val_window_MAE_mean", float(np.mean(window_maes))
            )
            trial.set_user_attr(
                "inner_fold_patient_MAEs", [float(v) for v in patient_maes]
            )
            return float(np.mean(patient_maes))

        sampler = optuna.samplers.TPESampler(seed=SEED + outer_fold)
        if model_name == "xgboost":
            pruner: optuna.pruners.BasePruner = optuna.pruners.MedianPruner(
                n_startup_trials=10,
                n_warmup_steps=2,
            )
        else:
            pruner = optuna.pruners.NopPruner()

        study = optuna.create_study(
            study_name=f"1d_10min_{model_name}_outer_{outer_fold:02d}",
            storage="sqlite:///" + os.path.join(model_dir, "optuna_study.db"),
            load_if_exists=True,
            direction="minimize",
            sampler=sampler,
            pruner=pruner,
        )

        existing_hash = study.user_attrs.get("experiment_hash")
        existing_outer_hash = study.user_attrs.get("outer_hash")
        existing_model = study.user_attrs.get("model")
        if existing_hash is None:
            if len(study.trials) > 0:
                raise RuntimeError(
                    f"Existing unsigned Optuna study in {model_dir}; "
                    "use a new output path."
                )
            study.set_user_attr("experiment_hash", EXPERIMENT_HASH)
            study.set_user_attr("outer_hash", outer_hash)
            study.set_user_attr("model", model_name)
        elif (
            existing_hash != EXPERIMENT_HASH
            or existing_outer_hash != outer_hash
            or existing_model != model_name
        ):
            raise RuntimeError(f"Optuna study identity mismatch in {model_dir}")

        top_up_study(
            study,
            objective,
            target_count=N_TRIALS_PER_OUTER[model_name],
        )

        if not any(
            t.state == optuna.trial.TrialState.COMPLETE for t in study.trials
        ):
            raise RuntimeError(
                f"No complete Optuna trial for {model_name}, outer {outer_fold}"
            )

        save_study_outputs(study, model_dir)
        best_params = dict(study.best_params)
        print(
            f"Outer {outer_fold} {model_name}: "
            f"best trial={study.best_trial.number}, "
            f"inner patient MAE={study.best_value:.4f}, params={best_params}"
        )

        seed_window_frames: List[pd.DataFrame] = []
        seed_patient_frames: List[pd.DataFrame] = []
        seed_metric_rows: List[Dict[str, Any]] = []

        for seed in FINAL_REFIT_SEEDS[model_name]:
            seed_dir = os.path.join(model_dir, f"seed_{seed}")
            os.makedirs(seed_dir, exist_ok=True)
            seed_marker = os.path.join(seed_dir, "SEED_COMPLETE.json")
            seed_window_csv = os.path.join(
                seed_dir, "outer_test_window_predictions.csv"
            )
            seed_patient_csv = os.path.join(
                seed_dir, "outer_test_patient_predictions.csv"
            )
            model_path = os.path.join(seed_dir, "fitted_model.joblib")

            if os.path.isfile(seed_marker):
                marker = read_json(seed_marker)
                expected_identity = {
                    "experiment_hash": EXPERIMENT_HASH,
                    "outer_hash": outer_hash,
                    "model": model_name,
                    "seed": int(seed),
                    "best_params_hash": stable_hash(best_params),
                }
                for key, value in expected_identity.items():
                    if marker.get(key) != value:
                        raise RuntimeError(f"Seed marker mismatch: {seed_marker}")
                if not os.path.isfile(seed_window_csv) or not os.path.isfile(
                    seed_patient_csv
                ):
                    raise RuntimeError(
                        f"Seed marker exists but CSVs are missing: {seed_dir}"
                    )
                window_df_seed = pd.read_csv(
                    seed_window_csv, dtype={"pid": str}
                )
                patient_df_seed = pd.read_csv(
                    seed_patient_csv, dtype={"pid": str}
                )
                seed_metrics = marker["metrics"]
                print(f"Outer {outer_fold} {model_name} seed {seed}: loaded.")
            else:
                model = fit_model(model_name, best_params, prepared_final, seed)
                X_test = prepared_final.load("X_eval")
                pred = np.asarray(model.predict(X_test), dtype=np.float32)

                window_df_seed = make_window_prediction_df(
                    prepared_final,
                    pred,
                    outer_fold=outer_fold,
                    model_name=model_name,
                    seed=seed,
                )
                patient_df_seed = patient_prediction_table(
                    window_df_seed["true_window"].values,
                    window_df_seed["pred_window"].values,
                    window_df_seed["pid"].values,
                    outer_fold=outer_fold,
                    model_name=model_name,
                    seed=seed,
                )

                window_metrics = regression_metrics(
                    window_df_seed["true_window"].values,
                    window_df_seed["pred_window"].values,
                )
                patient_metrics = regression_metrics(
                    patient_df_seed["true_patient"].values,
                    patient_df_seed["pred_patient"].values,
                )
                seed_metrics = {
                    "window": window_metrics,
                    "patient": patient_metrics,
                }

                window_df_seed.to_csv(seed_window_csv, index=False)
                patient_df_seed.to_csv(seed_patient_csv, index=False)
                atomic_joblib_dump(model, model_path)
                atomic_json_dump(
                    {
                        "experiment_hash": EXPERIMENT_HASH,
                        "outer_hash": outer_hash,
                        "model": model_name,
                        "seed": int(seed),
                        "best_params_hash": stable_hash(best_params),
                        "best_params": best_params,
                        "metrics": seed_metrics,
                        "model_path": model_path,
                    },
                    seed_marker,
                )
                del model, pred
                cleanup()

            seed_window_frames.append(window_df_seed)
            seed_patient_frames.append(patient_df_seed)
            seed_metric_rows.append(
                {
                    "outer_fold": outer_fold,
                    "model": model_name,
                    "seed": int(seed),
                    "patient_mae": seed_metrics["patient"]["mae"],
                    "patient_rmse": seed_metrics["patient"]["rmse"],
                    "patient_r2": seed_metrics["patient"]["r2"],
                    "window_mae": seed_metrics["window"]["mae"],
                    "window_rmse": seed_metrics["window"]["rmse"],
                    "window_r2": seed_metrics["window"]["r2"],
                }
            )

        pd.DataFrame(seed_metric_rows).to_csv(
            os.path.join(model_dir, "final_seed_metrics.csv"), index=False
        )
        all_seed_windows = pd.concat(seed_window_frames, ignore_index=True)
        all_seed_windows.to_csv(
            os.path.join(
                model_dir, "outer_test_window_predictions_all_seeds.csv"
            ),
            index=False,
        )
        all_seed_patients = pd.concat(seed_patient_frames, ignore_index=True)
        all_seed_patients.to_csv(
            os.path.join(
                model_dir, "outer_test_patient_predictions_all_seeds.csv"
            ),
            index=False,
        )

        ensemble_window = ensemble_seed_window_predictions(all_seed_windows)
        ensemble_patient = patient_from_ensemble_windows(ensemble_window)
        ensemble_window.to_csv(ensemble_window_csv, index=False)
        ensemble_patient.to_csv(ensemble_patient_csv, index=False)

        ensemble_window_metrics = regression_metrics(
            ensemble_window["true_window"].values,
            ensemble_window["pred_window_mean"].values,
        )
        ensemble_patient_metrics = regression_metrics(
            ensemble_patient["true_patient"].values,
            ensemble_patient["pred_patient"].values,
        )

        outer_summary = {
            "outer_fold": outer_fold,
            "model": model_name,
            "input": "10min_1D_feats_prefix_values_plus_missing_indicators",
            "n_outer_train_patients": len(outer_train_pids),
            "n_outer_test_patients": len(outer_test_pids),
            "n_outer_train_windows": int(prepared_final.meta["n_train_windows"]),
            "n_outer_test_windows": int(prepared_final.meta["n_eval_windows"]),
            "input_dim": int(prepared_final.meta["input_dim"]),
            "n_optuna_trials_target": int(N_TRIALS_PER_OUTER[model_name]),
            "best_trial": int(study.best_trial.number),
            "best_inner_patient_mae": float(study.best_value),
            "best_params_json": json.dumps(
                json_safe(best_params), sort_keys=True
            ),
            "final_refit_seeds": ";".join(
                map(str, FINAL_REFIT_SEEDS[model_name])
            ),
            "n_final_refit_seeds": len(FINAL_REFIT_SEEDS[model_name]),
            "outer_test_patient_mae": ensemble_patient_metrics["mae"],
            "outer_test_patient_rmse": ensemble_patient_metrics["rmse"],
            "outer_test_patient_r2": ensemble_patient_metrics["r2"],
            "outer_test_window_mae": ensemble_window_metrics["mae"],
            "outer_test_window_rmse": ensemble_window_metrics["rmse"],
            "outer_test_window_r2": ensemble_window_metrics["r2"],
        }
        outer_summary_rows.append(outer_summary)
        all_model_outer_windows[model_name].append(ensemble_window)
        all_model_outer_patients[model_name].append(ensemble_patient)

        atomic_json_dump(
            {
                "experiment_hash": EXPERIMENT_HASH,
                "outer_hash": outer_hash,
                "model": model_name,
                "outer_summary": outer_summary,
            },
            complete_marker,
        )
        pd.DataFrame(outer_summary_rows).to_csv(
            os.path.join(EXP_ROOT, "outer_fold_summary_running.csv"),
            index=False,
        )


# ============================================================
# 12. Pooled nested OOF results
# ============================================================
outer_summary_df = pd.DataFrame(outer_summary_rows).sort_values(
    ["model", "outer_fold"]
)
outer_summary_df.to_csv(
    os.path.join(EXP_ROOT, "outer_fold_summary.csv"), index=False
)

pooled_summary_rows: List[Dict[str, Any]] = []

for model_name in MODELS_TO_RUN:
    pooled_window = pd.concat(
        all_model_outer_windows[model_name], ignore_index=True
    )
    pooled_patient = pd.concat(
        all_model_outer_patients[model_name], ignore_index=True
    )

    expected_pids = set(patient_df["pid"].astype(str))
    observed_pids = set(pooled_patient["pid"].astype(str))
    if expected_pids != observed_pids:
        raise RuntimeError(
            f"{model_name}: pooled OOF patient mismatch. Missing="
            f"{sorted(expected_pids - observed_pids)[:20]}, unexpected="
            f"{sorted(observed_pids - expected_pids)[:20]}"
        )
    fold_counts = pooled_patient.groupby("pid")["outer_fold"].nunique()
    if not (fold_counts == 1).all():
        raise RuntimeError(
            f"{model_name}: a patient appears in multiple OOF test folds."
        )
    duplicate_windows = pooled_window.duplicated(
        subset=["pid", "window_idx"], keep=False
    )
    if duplicate_windows.any():
        raise RuntimeError(
            f"{model_name}: duplicated OOF patient-window rows."
        )

    pooled_window = pooled_window.sort_values(
        ["outer_fold", "pid", "window_idx"]
    ).reset_index(drop=True)
    pooled_patient = pooled_patient.sort_values(
        "abs_error", ascending=False
    ).reset_index(drop=True)

    pooled_window.to_csv(
        os.path.join(
            EXP_ROOT,
            f"FINAL_nested_oof_window_predictions_{model_name}.csv",
        ),
        index=False,
    )
    pooled_patient.to_csv(
        os.path.join(
            EXP_ROOT,
            f"FINAL_nested_oof_patient_predictions_{model_name}.csv",
        ),
        index=False,
    )

    window_metrics = regression_metrics(
        pooled_window["true_window"].values,
        pooled_window["pred_window_mean"].values,
    )
    patient_metrics = regression_metrics(
        pooled_patient["true_patient"].values,
        pooled_patient["pred_patient"].values,
    )

    summary_row = {
        "model": model_name,
        "input": "10min_1D_feats_prefix_values_plus_missing_indicators",
        "n_features_before_missing_indicators": len(feature_cols),
        "input_dim": len(feature_cols)
        * (2 if INCLUDE_MISSING_INDICATORS else 1),
        "n_patients": patient_metrics["n"],
        "n_windows": window_metrics["n"],
        "patient_mae": patient_metrics["mae"],
        "patient_rmse": patient_metrics["rmse"],
        "patient_r2": patient_metrics["r2"],
        "window_mae": window_metrics["mae"],
        "window_rmse": window_metrics["rmse"],
        "window_r2": window_metrics["r2"],
        "outer_fold_patient_mae_mean": float(
            outer_summary_df.loc[
                outer_summary_df["model"] == model_name,
                "outer_test_patient_mae",
            ].mean()
        ),
        "outer_fold_patient_mae_std": float(
            outer_summary_df.loc[
                outer_summary_df["model"] == model_name,
                "outer_test_patient_mae",
            ].std(ddof=0)
        ),
    }
    pooled_summary_rows.append(summary_row)

    fig, ax = plt.subplots(figsize=(6.5, 6.5))
    ax.scatter(
        pooled_patient["true_patient"],
        pooled_patient["pred_patient"],
        alpha=0.7,
    )
    low = float(
        min(
            pooled_patient["true_patient"].min(),
            pooled_patient["pred_patient"].min(),
        )
    )
    high = float(
        max(
            pooled_patient["true_patient"].max(),
            pooled_patient["pred_patient"].max(),
        )
    )
    ax.plot([low, high], [low, high], linestyle="--")
    ax.set_xlabel("True patient-level PMA from 10-min windows (weeks)")
    ax.set_ylabel("Predicted patient-level PMA (weeks)")
    ax.set_title(
        f"{model_name}: 1D 10-min nested OOF\n"
        f"MAE={patient_metrics['mae']:.3f}, "
        f"RMSE={patient_metrics['rmse']:.3f}, "
        f"R²={patient_metrics['r2']:.3f}"
    )
    fig.tight_layout()
    fig.savefig(
        os.path.join(
            EXP_ROOT, f"FINAL_patient_true_vs_pred_{model_name}.png"
        ),
        dpi=300,
        bbox_inches="tight",
    )
    plt.close(fig)

pooled_summary_df = pd.DataFrame(pooled_summary_rows)
pooled_summary_df.to_csv(
    os.path.join(EXP_ROOT, "FINAL_1d_baseline_summary.csv"), index=False
)
atomic_json_dump(
    {"experiment_hash": EXPERIMENT_HASH, "summary": pooled_summary_rows},
    os.path.join(EXP_ROOT, "FINAL_1d_baseline_summary.json"),
)


# ============================================================
# 13. Paired comparison using the CNN's common patient ground truth
# ============================================================
if os.path.isfile(MAIN_CNN_PATIENT_OOF_CSV):
    cnn = pd.read_csv(MAIN_CNN_PATIENT_OOF_CSV, dtype={"pid": str})
    cnn["pid"] = cnn["pid"].map(normalize_pid_text)
    cnn_pred_col = (
        "pred_patient_mean"
        if "pred_patient_mean" in cnn.columns
        else "pred_patient"
    )
    required_cnn = {"pid", "true_patient", cnn_pred_col}

    if required_cnn.issubset(cnn.columns):
        cnn_small = cnn[["pid", "true_patient", cnn_pred_col]].copy()
        cnn_small = cnn_small.rename(
            columns={
                "true_patient": "true_patient_common_CNN_definition",
                cnn_pred_col: "pred_patient_cnn",
            }
        )
        cnn_small["abs_error_cnn"] = np.abs(
            cnn_small["pred_patient_cnn"]
            - cnn_small["true_patient_common_CNN_definition"]
        )

        comparison = cnn_small.copy()
        common_truth_summary: List[Dict[str, Any]] = []

        for model_name in MODELS_TO_RUN:
            baseline = pd.read_csv(
                os.path.join(
                    EXP_ROOT,
                    f"FINAL_nested_oof_patient_predictions_{model_name}.csv",
                ),
                dtype={"pid": str},
            )
            baseline["pid"] = baseline["pid"].map(normalize_pid_text)
            baseline = baseline[
                ["pid", "true_patient", "pred_patient", "abs_error"]
            ].rename(
                columns={
                    "true_patient": f"true_patient_native_1d_{model_name}",
                    "pred_patient": f"pred_patient_{model_name}",
                    "abs_error": f"abs_error_native_1d_{model_name}",
                }
            )
            comparison = comparison.merge(baseline, on="pid", how="inner")

            common_abs_error_col = f"abs_error_common_truth_{model_name}"
            comparison[common_abs_error_col] = np.abs(
                comparison[f"pred_patient_{model_name}"]
                - comparison["true_patient_common_CNN_definition"]
            )
            comparison[
                f"abs_error_difference_{model_name}_minus_cnn"
            ] = comparison[common_abs_error_col] - comparison["abs_error_cnn"]

            common_metrics = regression_metrics(
                comparison["true_patient_common_CNN_definition"].values,
                comparison[f"pred_patient_{model_name}"].values,
            )
            common_truth_summary.append(
                {
                    "model": model_name,
                    "truth_definition": "main_CNN_patient_true_PMA",
                    "patient_mae": common_metrics["mae"],
                    "patient_rmse": common_metrics["rmse"],
                    "patient_r2": common_metrics["r2"],
                    "n_patients": common_metrics["n"],
                }
            )

        comparison.to_csv(
            os.path.join(
                EXP_ROOT, "FINAL_paired_patient_comparison_with_CNN.csv"
            ),
            index=False,
        )
        pd.DataFrame(common_truth_summary).to_csv(
            os.path.join(
                EXP_ROOT, "FINAL_1d_metrics_using_common_CNN_patient_truth.csv"
            ),
            index=False,
        )
    else:
        warnings.warn(
            "CNN patient OOF CSV exists but expected columns are missing; "
            "paired comparison was skipped."
        )
else:
    warnings.warn(
        "MAIN_CNN_PATIENT_OOF_CSV was not found. The 1D baseline completed, "
        "but paired CNN comparison was skipped."
    )

print("\n" + "=" * 110)
print("1D 10-MIN BASELINE NESTED CV COMPLETE")
print(pooled_summary_df.to_string(index=False))
print("Outputs saved to:", EXP_ROOT)
print("=" * 110)

In [ ]:
EXP_ROOT = os.path.join(
    PROJECT_ROOT,
    "result_nested",
    "baseline_1d_10min_ridge_xgboost_nestedcv_optuna",
)

In [ ]:
# ============================================================
# GA-bin post-hoc analysis for completed 1D Ridge + XGBoost
# Paste directly after the finished 1D baseline cell.
#
# Generates TWO complete versions:
#   1) ALL PMA bins, without patient-count filtering
#   2) FILTERED PMA bins, retaining bins with >= 5 patients
#
# It does NOT retrain any model.
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score

MIN_PATIENTS_PER_PMA_BIN = 5
PMA_BIN_WIDTH = 0.5

GA_DIR = os.path.join(EXP_ROOT, "ga_bin_analysis")
os.makedirs(GA_DIR, exist_ok=True)

MODELS = ("ridge", "xgboost")

GROUP_ORDER = [
    "EP < 26 wk",
    "VP [26,29) wk",
    "MP [29,32) wk",
    "LP [32,36) wk",
]

GROUP_LABELS = {
    "EP < 26 wk": "Extreme preterm (<26 wk)",
    "VP [26,29) wk": "Very preterm ([26,29) wk)",
    "MP [29,32) wk": "Moderate preterm ([29,32) wk)",
    "LP [32,36) wk": "Late preterm ([32,36) wk)",
}

GROUP_COLORS = {
    "EP < 26 wk": "#b24726",
    "VP [26,29) wk": "#d49a27",
    "MP [29,32) wk": "#4c9a8a",
    "LP [32,36) wk": "#4a78c2",
}


def norm_pid(value):
    text = str(value)
    if text.endswith(".0") and text[:-2].isdigit():
        return text[:-2]
    return text


def assign_ga_group(ga):
    if pd.isna(ga):
        return np.nan
    ga = float(ga)
    if ga < 26:
        return "EP < 26 wk"
    if ga < 29:
        return "VP [26,29) wk"
    if ga < 32:
        return "MP [29,32) wk"
    if ga < 36:
        return "LP [32,36) wk"
    return np.nan


def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    valid = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[valid]
    y_pred = y_pred[valid]

    if len(y_true) == 0:
        return {
            "mae": np.nan,
            "rmse": np.nan,
            "r2": np.nan,
            "bias": np.nan,
            "n": 0,
        }

    error = y_pred - y_true
    r2 = (
        float(r2_score(y_true, y_pred))
        if len(y_true) > 1 and not np.allclose(y_true, y_true[0])
        else np.nan
    )

    return {
        "mae": float(np.mean(np.abs(error))),
        "rmse": float(np.sqrt(np.mean(error ** 2))),
        "r2": r2,
        "bias": float(np.mean(error)),
        "n": int(len(y_true)),
    }


def make_pma_bin_table(window_df):
    parts = []

    for group in GROUP_ORDER:
        group_df = window_df[window_df["ga_group"] == group].copy()
        if group_df.empty:
            continue

        pma_min = float(group_df["true_window"].min())
        pma_max = float(group_df["true_window"].max())

        lower = np.floor(pma_min / PMA_BIN_WIDTH) * PMA_BIN_WIDTH
        upper = np.ceil(pma_max / PMA_BIN_WIDTH) * PMA_BIN_WIDTH

        edges = np.arange(
            lower,
            upper + PMA_BIN_WIDTH + 1e-9,
            PMA_BIN_WIDTH,
        )

        if len(edges) < 2:
            continue

        group_df["pma_bin"] = pd.cut(
            group_df["true_window"],
            bins=edges,
            include_lowest=True,
            right=True,
        )

        summary = (
            group_df.groupby("pma_bin", observed=True)
            .agg(
                true_mid=("true_window", "median"),
                pred_med=("pred_window_mean", "median"),
                pred_q1=(
                    "pred_window_mean",
                    lambda x: float(np.nanquantile(x, 0.25)),
                ),
                pred_q3=(
                    "pred_window_mean",
                    lambda x: float(np.nanquantile(x, 0.75)),
                ),
                mean_error=("error_window", "mean"),
                mae=("abs_error_window", "mean"),
                n_windows=("pred_window_mean", "size"),
                n_patients=("pid", "nunique"),
            )
            .reset_index()
        )

        summary["pma_bin"] = summary["pma_bin"].astype(str)
        summary["ga_group"] = group
        summary["ga_label"] = GROUP_LABELS[group]
        summary["keep_min5"] = (
            summary["n_patients"] >= MIN_PATIENTS_PER_PMA_BIN
        )
        parts.append(summary)

    if not parts:
        return pd.DataFrame()

    return pd.concat(parts, ignore_index=True)


def plot_binned_trajectory(bin_df, model, output_dir, suffix, title_suffix):
    """Save combined binned trajectory plus patient-count panel."""
    plot_df = bin_df.dropna(
        subset=["true_mid", "pred_med", "pred_q1", "pred_q3"]
    ).copy() if not bin_df.empty else pd.DataFrame()

    if plot_df.empty:
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.axis("off")
        ax.text(
            0.5, 0.5,
            f"{model}: no valid PMA bins\n{title_suffix}",
            ha="center", va="center", fontsize=14,
            transform=ax.transAxes,
        )
        fig.savefig(
            os.path.join(output_dir, f"02_ga_bin_binned_trajectory_{suffix}.png"),
            dpi=300, bbox_inches="tight",
        )
        plt.close(fig)
        print(f"Saved empty-result placeholder: {model}, {suffix}")
        return

    fig, (ax1, ax2) = plt.subplots(
        2,
        1,
        figsize=(10, 9),
        sharex=True,
        gridspec_kw={"height_ratios": [3, 1], "hspace": 0.08},
    )

    axis_min = float(min(plot_df["true_mid"].min(), plot_df["pred_med"].min()))
    axis_max = float(max(plot_df["true_mid"].max(), plot_df["pred_med"].max()))

    ax1.plot(
        [axis_min, axis_max],
        [axis_min, axis_max],
        linestyle="--",
        color="black",
        linewidth=1.5,
        label="Reference (prediction = PMA)",
    )

    for group in GROUP_ORDER:
        group_bins = plot_df[
            plot_df["ga_group"] == group
        ].sort_values("true_mid")

        if group_bins.empty:
            continue

        x = group_bins["true_mid"].to_numpy(dtype=float)
        pred_med = group_bins["pred_med"].to_numpy(dtype=float)
        pred_q1 = group_bins["pred_q1"].to_numpy(dtype=float)
        pred_q3 = group_bins["pred_q3"].to_numpy(dtype=float)
        n_patients = group_bins["n_patients"].to_numpy(dtype=float)

        ax1.plot(
            x,
            pred_med,
            marker="o",
            linewidth=2.4,
            color=GROUP_COLORS[group],
            label=GROUP_LABELS[group],
        )
        ax1.fill_between(
            x,
            pred_q1,
            pred_q3,
            color=GROUP_COLORS[group],
            alpha=0.12,
        )

        ax2.plot(
            x,
            n_patients,
            marker="o",
            linewidth=2.0,
            color=GROUP_COLORS[group],
            label=group,
        )

    ax1.set_ylabel("Median predicted maturational age (weeks)")
    ax1.set_title(
        f"{model}: binned median trajectory by birth-GA group\n{title_suffix}"
    )
    ax1.legend(loc="upper left", fontsize=9)
    ax1.grid(True, alpha=0.25)

    ax2.set_xlabel("Median PMA within 0.5-week bin")
    ax2.set_ylabel("Patient count")
    ax2.grid(True, alpha=0.25)

    fig.tight_layout()
    fig.savefig(
        os.path.join(
            output_dir,
            f"02_ga_bin_binned_trajectory_{suffix}.png",
        ),
        dpi=300,
        bbox_inches="tight",
    )
    plt.close(fig)


def plot_four_panel(bin_df, model, output_dir, suffix, title_suffix):
    """Save one binned trajectory panel for each GA group."""
    fig, axes = plt.subplots(2, 2, figsize=(11, 9), sharex=False, sharey=False)
    axes = axes.flatten()
    fig.suptitle(
        f"{model}: binned trajectories by birth-GA group\n{title_suffix}",
        fontsize=15,
    )

    for ax, group in zip(axes, GROUP_ORDER):
        group_bins = bin_df[
            bin_df["ga_group"] == group
        ].sort_values("true_mid")

        ax.set_title(GROUP_LABELS[group], fontsize=12, fontweight="bold")
        ax.set_xlabel("Median PMA within bin")
        ax.set_ylabel("Median predicted age")
        ax.grid(True, alpha=0.25)

        if group_bins.empty:
            ax.text(
                0.5,
                0.5,
                "No valid bins",
                ha="center",
                va="center",
                transform=ax.transAxes,
            )
            continue

        x = group_bins["true_mid"].to_numpy(dtype=float)
        pred_med = group_bins["pred_med"].to_numpy(dtype=float)
        pred_q1 = group_bins["pred_q1"].to_numpy(dtype=float)
        pred_q3 = group_bins["pred_q3"].to_numpy(dtype=float)

        axis_min = float(min(x.min(), pred_med.min()))
        axis_max = float(max(x.max(), pred_med.max()))

        ax.plot(
            [axis_min, axis_max],
            [axis_min, axis_max],
            linestyle="--",
            color="black",
            linewidth=1.2,
        )
        ax.plot(
            x,
            pred_med,
            marker="o",
            linewidth=2.4,
            color=GROUP_COLORS[group],
        )
        ax.fill_between(
            x,
            pred_q1,
            pred_q3,
            color=GROUP_COLORS[group],
            alpha=0.12,
        )

        n_bins = int(len(group_bins))
        min_patients = int(group_bins["n_patients"].min())
        max_patients = int(group_bins["n_patients"].max())
        ax.text(
            0.97,
            0.05,
            f"bins = {n_bins}\npatients/bin = {min_patients}–{max_patients}",
            transform=ax.transAxes,
            ha="right",
            va="bottom",
            fontsize=9,
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.75),
        )

    fig.tight_layout(rect=[0, 0, 1, 0.94])
    fig.savefig(
        os.path.join(
            output_dir,
            f"03_ga_bin_4panel_binned_trajectory_{suffix}.png",
        ),
        dpi=300,
        bbox_inches="tight",
    )
    plt.close(fig)


# ============================================================
# 1. Extract birth GA for the 1D cohort
#    Preferred: reuse arrays still in memory from the 1D baseline.
#    Fallback: reload the filtered 10-min dataframe from DATA_PKL.
# ============================================================
GA_FEATURE = "feats__ga_w"
ID_FEATURE = "ids__uid"

pid_to_ga = {}

if all(name in globals() for name in ["X_RAW", "PID_ALL", "feature_cols"]):
    if GA_FEATURE not in feature_cols:
        raise KeyError(f"{GA_FEATURE} is absent from feature_cols.")

    ga_idx = feature_cols.index(GA_FEATURE)
    ga_values_all = np.asarray(X_RAW[:, ga_idx], dtype=float)
    pid_values_all = np.asarray(PID_ALL).astype(str)

    ga_source = pd.DataFrame(
        {
            "pid": [norm_pid(x) for x in pid_values_all],
            "ga_birth": ga_values_all,
        }
    )
    ga_source = ga_source[np.isfinite(ga_source["ga_birth"])].copy()
    pid_to_ga = (
        ga_source.groupby("pid", sort=False)["ga_birth"]
        .median()
        .astype(float)
        .to_dict()
    )
    ga_source_mode = "reused_X_RAW_and_PID_ALL"

else:
    if "DATA_PKL" not in globals():
        raise RuntimeError(
            "Neither in-memory 1D arrays nor DATA_PKL are available. "
            "Run this cell after the completed 1D baseline cell."
        )
    if not os.path.isfile(DATA_PKL):
        raise FileNotFoundError(f"DATA_PKL does not exist: {DATA_PKL}")

    import pickle

    with open(DATA_PKL, "rb") as file_handle:
        raw_df = pickle.load(file_handle)

    required_ga_columns = {ID_FEATURE, GA_FEATURE}
    missing_ga_columns = required_ga_columns - set(raw_df.columns)
    if missing_ga_columns:
        raise KeyError(
            f"Filtered 1D dataframe is missing: {sorted(missing_ga_columns)}"
        )

    ga_source = raw_df[[ID_FEATURE, GA_FEATURE]].copy()
    del raw_df

    ga_source["pid"] = ga_source[ID_FEATURE].map(norm_pid)
    ga_source["ga_birth"] = pd.to_numeric(
        ga_source[GA_FEATURE], errors="coerce"
    )
    ga_source = ga_source[np.isfinite(ga_source["ga_birth"])].copy()

    pid_to_ga = (
        ga_source.groupby("pid", sort=False)["ga_birth"]
        .median()
        .astype(float)
        .to_dict()
    )
    ga_source_mode = "reloaded_filtered_10min_dataframe"

if not pid_to_ga:
    raise RuntimeError("No valid birth-GA values were extracted.")

print(
    "Birth GA extracted for",
    len(pid_to_ga),
    "patients using",
    ga_source_mode,
)


# ============================================================
# 2. Analyze completed Ridge and XGBoost OOF predictions
# ============================================================
all_group_tables = []
all_bin_tables = []

for model in MODELS:
    print("\n" + "=" * 100)
    print("GA-bin analysis:", model)
    print("=" * 100)

    model_dir = os.path.join(GA_DIR, model)
    os.makedirs(model_dir, exist_ok=True)

    patient_csv = os.path.join(
        EXP_ROOT,
        f"FINAL_nested_oof_patient_predictions_{model}.csv",
    )
    window_csv = os.path.join(
        EXP_ROOT,
        f"FINAL_nested_oof_window_predictions_{model}.csv",
    )

    if not os.path.isfile(patient_csv):
        raise FileNotFoundError(patient_csv)
    if not os.path.isfile(window_csv):
        raise FileNotFoundError(window_csv)

    patient_df = pd.read_csv(patient_csv, dtype={"pid": str})
    window_df = pd.read_csv(window_csv, dtype={"pid": str})

    patient_df["pid"] = patient_df["pid"].map(norm_pid)
    window_df["pid"] = window_df["pid"].map(norm_pid)

    if "pred_patient" not in patient_df.columns:
        if "pred_patient_mean" in patient_df.columns:
            patient_df = patient_df.rename(
                columns={"pred_patient_mean": "pred_patient"}
            )
        else:
            raise KeyError(f"No patient prediction column in {patient_csv}")

    if "pred_window_mean" not in window_df.columns:
        if "pred_window" in window_df.columns:
            window_df = window_df.rename(
                columns={"pred_window": "pred_window_mean"}
            )
        else:
            raise KeyError(f"No window prediction column in {window_csv}")

    for current_df in (patient_df, window_df):
        current_df["ga_birth"] = current_df["pid"].map(pid_to_ga)
        current_df["ga_group"] = current_df["ga_birth"].apply(assign_ga_group)
        current_df["model"] = model

    patient_df["error_patient"] = (
        patient_df["pred_patient"] - patient_df["true_patient"]
    )
    patient_df["abs_error_patient"] = patient_df["error_patient"].abs()

    window_df["error_window"] = (
        window_df["pred_window_mean"] - window_df["true_window"]
    )
    window_df["abs_error_window"] = window_df["error_window"].abs()

    missing_ga = sorted(
        patient_df.loc[patient_df["ga_birth"].isna(), "pid"].unique()
    )
    if missing_ga:
        pd.DataFrame({"pid": missing_ga}).to_csv(
            os.path.join(model_dir, "patients_missing_birth_ga.csv"),
            index=False,
        )
        print("Patients missing birth GA:", len(missing_ga))

    patient_df = patient_df.dropna(subset=["ga_group"]).copy()
    window_df = window_df.dropna(subset=["ga_group"]).copy()

    patient_df.to_csv(
        os.path.join(model_dir, "ga_bin_patient_predictions.csv"),
        index=False,
    )
    window_df.to_csv(
        os.path.join(model_dir, "ga_bin_window_predictions.csv"),
        index=False,
    )

    # --------------------------------------------------------
    # 2A. Metrics by birth-GA category
    # --------------------------------------------------------
    group_rows = []

    for group in GROUP_ORDER:
        patient_group = patient_df[patient_df["ga_group"] == group]
        window_group = window_df[window_df["ga_group"] == group]

        patient_metrics = regression_metrics(
            patient_group["true_patient"],
            patient_group["pred_patient"],
        )
        window_metrics = regression_metrics(
            window_group["true_window"],
            window_group["pred_window_mean"],
        )

        group_rows.append(
            {
                "model": model,
                "ga_group": group,
                "ga_label": GROUP_LABELS[group],
                "n_patients": int(patient_group["pid"].nunique()),
                "n_windows": int(len(window_group)),
                "patient_mae": patient_metrics["mae"],
                "patient_rmse": patient_metrics["rmse"],
                "patient_r2": patient_metrics["r2"],
                "patient_bias": patient_metrics["bias"],
                "window_mae": window_metrics["mae"],
                "window_rmse": window_metrics["rmse"],
                "window_r2": window_metrics["r2"],
                "window_bias": window_metrics["bias"],
            }
        )

    group_table = pd.DataFrame(group_rows)
    group_table.to_csv(
        os.path.join(model_dir, "ga_bin_group_summary.csv"),
        index=False,
    )
    all_group_tables.append(group_table)

    # --------------------------------------------------------
    # 2B. PMA-bin tables: save BOTH all and filtered versions
    # --------------------------------------------------------
    all_bins = make_pma_bin_table(window_df)
    if all_bins.empty:
        raise RuntimeError(f"No PMA bins generated for {model}.")

    all_bins["model"] = model
    filtered_bins = all_bins[all_bins["keep_min5"]].copy()

    all_bins.to_csv(
        os.path.join(model_dir, "ga_bin_pma_bins_all.csv"),
        index=False,
    )
    filtered_bins.to_csv(
        os.path.join(
            model_dir,
            f"ga_bin_pma_bins_filtered_min{MIN_PATIENTS_PER_PMA_BIN}patients.csv",
        ),
        index=False,
    )
    all_bin_tables.append(all_bins)

    # --------------------------------------------------------
    # 2C. Scatter plot using all OOF windows
    # --------------------------------------------------------
    fig, ax = plt.subplots(figsize=(8, 8))

    for group in GROUP_ORDER:
        group_windows = window_df[window_df["ga_group"] == group]
        if group_windows.empty:
            continue

        ax.scatter(
            group_windows["true_window"],
            group_windows["pred_window_mean"],
            s=18,
            alpha=0.18,
            color=GROUP_COLORS[group],
            label=group,
        )

    axis_min = float(
        min(window_df["true_window"].min(), window_df["pred_window_mean"].min())
    )
    axis_max = float(
        max(window_df["true_window"].max(), window_df["pred_window_mean"].max())
    )
    reference = np.linspace(axis_min, axis_max, 300)

    ax.plot(reference, reference, linestyle="--", color="black")
    ax.fill_between(
        reference,
        reference - 1,
        reference + 1,
        color="gray",
        alpha=0.12,
    )
    ax.set_xlabel("Post-menstrual age (weeks)")
    ax.set_ylabel("Predicted maturational age (weeks)")
    ax.set_title(f"{model}: all OOF windows by birth-GA group")
    ax.legend(loc="upper left")
    ax.grid(True, alpha=0.25)
    fig.tight_layout()
    fig.savefig(
        os.path.join(model_dir, "01_ga_bin_combined_scatter_all_windows.png"),
        dpi=300,
        bbox_inches="tight",
    )
    plt.close(fig)

    # --------------------------------------------------------
    # 2D. Save BOTH complete plot sets
    # --------------------------------------------------------
    plot_binned_trajectory(
        bin_df=all_bins,
        model=model,
        output_dir=model_dir,
        suffix="all",
        title_suffix="All 0.5-week PMA bins",
    )
    plot_four_panel(
        bin_df=all_bins,
        model=model,
        output_dir=model_dir,
        suffix="all",
        title_suffix="All 0.5-week PMA bins",
    )

    plot_binned_trajectory(
        bin_df=filtered_bins,
        model=model,
        output_dir=model_dir,
        suffix=f"filtered_min{MIN_PATIENTS_PER_PMA_BIN}patients",
        title_suffix=(
            f"Only bins with at least {MIN_PATIENTS_PER_PMA_BIN} patients"
        ),
    )
    plot_four_panel(
        bin_df=filtered_bins,
        model=model,
        output_dir=model_dir,
        suffix=f"filtered_min{MIN_PATIENTS_PER_PMA_BIN}patients",
        title_suffix=(
            f"Only bins with at least {MIN_PATIENTS_PER_PMA_BIN} patients"
        ),
    )

    print(
        f"Finished {model}: all bins={len(all_bins)}, "
        f"filtered bins={len(filtered_bins)}"
    )


# ============================================================
# 3. Combined outputs across Ridge and XGBoost
# ============================================================
combined_group_summary = pd.concat(all_group_tables, ignore_index=True)
combined_all_bins = pd.concat(all_bin_tables, ignore_index=True)
combined_filtered_bins = combined_all_bins[
    combined_all_bins["keep_min5"]
].copy()

combined_group_summary.to_csv(
    os.path.join(
        GA_DIR,
        "FINAL_ga_bin_group_summary_all_1d_baselines.csv",
    ),
    index=False,
)
combined_all_bins.to_csv(
    os.path.join(
        GA_DIR,
        "FINAL_ga_bin_pma_bins_all_1d_baselines.csv",
    ),
    index=False,
)
combined_filtered_bins.to_csv(
    os.path.join(
        GA_DIR,
        f"FINAL_ga_bin_pma_bins_filtered_min{MIN_PATIENTS_PER_PMA_BIN}patients.csv",
    ),
    index=False,
)

print("\n" + "=" * 100)
print("GA-BIN ANALYSIS COMPLETE")
print("Saved to:", GA_DIR)
print("Both ALL and FILTERED_MIN5 versions were generated.")
print(
    combined_group_summary[
        [
            "model",
            "ga_group",
            "n_patients",
            "patient_mae",
            "patient_rmse",
            "patient_r2",
            "patient_bias",
        ]
    ].to_string(index=False)
)
print("=" * 100)